# Análisis de resultados — STGNN Temperatura DMQ
**Modelo:** A3T-GCN (comparación con DCRNN cuando esté disponible)
**Escenarios:** corto (h=3), medio (h=48), largo (h=72)

Este notebook produce tablas, gráficas comparativas, análisis de error por
estación y un *forecasting* sobre marzo 2026. Ajusta las rutas en la primera
celda si tu estructura de carpetas difiere.


In [ ]:
%matplotlib inline
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# fallback de display para correr también fuera de Jupyter
try:
    display
except NameError:
    def display(x): print(x)

# ------------------- RUTAS (ajusta si difiere) -------------------
ART_DIR   = "artefactos"
RES_A3T   = "resultados/a3tgcn"
RES_DCRNN = "resultados/dcrnn"          # opcional (si ya corriste DCRNN)
FIG_DIR   = "resultados/figuras"
os.makedirs(FIG_DIR, exist_ok=True)

ESCENARIOS = ["corto", "medio", "largo"]
COLORS = {"corto": "#2a9d8f", "medio": "#e9a12a", "largo": "#e76f51"}
ESTACION_FORECAST = "Belisario"         # estación para el forecasting detallado

plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})


## 1. Carga de artefactos, métricas y utilidades

In [ ]:
def cargar_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

met_a3t   = cargar_json(os.path.join(RES_A3T, "metricas_a3tgcn.json"))
tiempos   = np.load(os.path.join(ART_DIR, "tiempos.npy"))
splits    = cargar_json(os.path.join(ART_DIR, "splits.json"))
estaciones = cargar_json(os.path.join(ART_DIR, "station_order.json"))
test_a, test_b = splits["test"]

print("Estaciones:", estaciones)
print("Ventana TEST:", str(tiempos[test_a])[:16], "->", str(tiempos[test_b-1])[:16],
      f"({test_b-test_a:,} pasos)")

def cargar_preds(res_dir, prefijo, esc, met):
    """Devuelve (pred, true, timestamps, L, h). pred/true en °C, forma (S, N)."""
    h = met[esc]["config"]["horizon"]; L = met[esc]["config"]["seq_len"]
    tag = f"{prefijo}_{esc}_h{h}"
    pred = np.load(os.path.join(res_dir, f"{tag}_pred_test.npy"))
    true = np.load(os.path.join(res_dir, f"{tag}_true_test.npy"))
    S = pred.shape[0]
    start = test_a + L + h - 1          # índice global del 1er target de test
    ts = tiempos[start:start + S]
    return pred, true, ts, L, h

PREDS_OK = all(
    os.path.exists(os.path.join(RES_A3T,
        f"a3tgcn_{e}_h{met_a3t[e]['config']['horizon']}_pred_test.npy"))
    for e in ESCENARIOS)
print("Arrays de predicción disponibles:", PREDS_OK)


## 2. Tabla de métricas globales (A3T-GCN)

In [ ]:
filas = []
for esc in ESCENARIOS:
    g = met_a3t[esc]["global"]; c = met_a3t[esc]["config"]
    filas.append({"escenario": esc, "horizonte_h": c["horizon"], "seq_len": c["seq_len"],
                  "MAE (°C)": g["MAE"], "RMSE (°C)": g["RMSE"], "R²": g["R2"],
                  "MAPE (%)": g["MAPE"], "epochs": met_a3t[esc]["epochs_corridos"]})
tabla_global = pd.DataFrame(filas).set_index("escenario").round(4)
tabla_global.to_csv(os.path.join(FIG_DIR, "tabla_global_a3tgcn.csv"))
display(tabla_global)


## 3. Métricas por estación

In [ ]:
def tabla_por_estacion(met, metrica):
    d = {esc: {est: met[esc]["por_estacion"][est][metrica] for est in estaciones}
         for esc in ESCENARIOS}
    return pd.DataFrame(d).round(4)

tabla_mae_est = tabla_por_estacion(met_a3t, "MAE")
tabla_r2_est  = tabla_por_estacion(met_a3t, "R2")
print("MAE por estación (°C):"); display(tabla_mae_est)
print("\nR² por estación:");     display(tabla_r2_est)
tabla_mae_est.to_csv(os.path.join(FIG_DIR, "mae_por_estacion_a3tgcn.csv"))


## 4. Curva de degradación (error vs horizonte)

In [ ]:
hs   = [met_a3t[e]["config"]["horizon"] for e in ESCENARIOS]
mae  = [met_a3t[e]["global"]["MAE"]  for e in ESCENARIOS]
rmse = [met_a3t[e]["global"]["RMSE"] for e in ESCENARIOS]
r2   = [met_a3t[e]["global"]["R2"]   for e in ESCENARIOS]

fig, ax1 = plt.subplots(figsize=(7.5, 4.6))
ax1.plot(hs, mae,  "o-", color="#264653", label="MAE")
ax1.plot(hs, rmse, "s--", color="#e76f51", label="RMSE")
ax1.set_xlabel("Horizonte de predicción (h)"); ax1.set_ylabel("Error (°C)")
ax1.set_xticks(hs)
ax2 = ax1.twinx(); ax2.plot(hs, r2, "^-", color="#2a9d8f", label="R²")
ax2.set_ylabel("R²"); ax2.grid(False)
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="center right")
ax1.set_title("Degradación del desempeño vs horizonte — A3T-GCN")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "degradacion.png"), bbox_inches="tight")
plt.show()


## 5. Barras comparativas de error por escenario

In [ ]:
x = np.arange(len(ESCENARIOS)); w = 0.35
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.bar(x - w/2, mae,  w, label="MAE",  color="#264653")
ax.bar(x + w/2, rmse, w, label="RMSE", color="#e76f51")
ax.set_xticks(x)
ax.set_xticklabels([f"{e}\n(h={met_a3t[e]['config']['horizon']})" for e in ESCENARIOS])
ax.set_ylabel("Error (°C)"); ax.set_title("MAE y RMSE por escenario — A3T-GCN")
ax.legend()
for i, (m, r) in enumerate(zip(mae, rmse)):
    ax.text(i - w/2, m + 0.02, f"{m:.2f}", ha="center", va="bottom", fontsize=9)
    ax.text(i + w/2, r + 0.02, f"{r:.2f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "barras_error.png"), bbox_inches="tight")
plt.show()


## 6. Heatmap de MAE por estación y horizonte

In [ ]:
data = tabla_mae_est[ESCENARIOS].values
fig, ax = plt.subplots(figsize=(6.2, 4.6))
im = ax.imshow(data, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(ESCENARIOS)))
ax.set_xticklabels([f"{e}\nh={met_a3t[e]['config']['horizon']}" for e in ESCENARIOS])
ax.set_yticks(range(len(estaciones))); ax.set_yticklabels(estaciones)
thr = data.max() * 0.7
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j, i, f"{data[i, j]:.2f}", ha="center", va="center", fontsize=9,
                color="black" if data[i, j] < thr else "white")
fig.colorbar(im, label="MAE (°C)")
ax.set_title("MAE por estación y horizonte — A3T-GCN")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "heatmap_mae.png"), bbox_inches="tight")
plt.show()


## 7. Dispersión observado vs predicho (TEST)

In [ ]:
if PREDS_OK:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))
    rng = np.random.default_rng(0)
    for ax, esc in zip(axes, ESCENARIOS):
        pred, true, ts, L, h = cargar_preds(RES_A3T, "a3tgcn", esc, met_a3t)
        yt, yp = true.ravel(), pred.ravel()
        idx = rng.choice(len(yt), size=min(6000, len(yt)), replace=False)
        ax.scatter(yt[idx], yp[idx], s=6, alpha=0.25, color=COLORS[esc])
        lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
        ax.plot(lims, lims, "k--", lw=1)
        ax.set_xlabel("Observado (°C)"); ax.set_ylabel("Predicho (°C)")
        ax.set_title(f"{esc} (h={h}) — R²={met_a3t[esc]['global']['R2']:.3f}")
    plt.suptitle("Observado vs Predicho — A3T-GCN (TEST)", y=1.03)
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "scatter_obs_pred.png"), bbox_inches="tight")
    plt.show()
else:
    print("Faltan arrays de predicción; omito la dispersión.")


## 8. Distribución de residuales por horizonte

In [ ]:
if PREDS_OK:
    res = []
    for esc in ESCENARIOS:
        pred, true, ts, L, h = cargar_preds(RES_A3T, "a3tgcn", esc, met_a3t)
        res.append((pred - true).ravel())
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    bp = ax.boxplot(res, showfliers=False, patch_artist=True,
                    tick_labels=[f"{e}\nh={met_a3t[e]['config']['horizon']}" for e in ESCENARIOS])
    for patch, esc in zip(bp["boxes"], ESCENARIOS):
        patch.set_facecolor(COLORS[esc]); patch.set_alpha(0.75)
    ax.axhline(0, color="k", lw=1)
    ax.set_ylabel("Residual (predicho − observado) °C")
    ax.set_title("Distribución de residuales — A3T-GCN")
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "residuales.png"), bbox_inches="tight")
    plt.show()
    for esc, r in zip(ESCENARIOS, res):
        print(f"{esc:6s} | sesgo medio {r.mean():+.3f} °C | std {r.std():.3f} °C")
else:
    print("Faltan arrays de predicción; omito residuales.")


## 9. Forecasting — Marzo 2026
Observado vs predicho para cada horizonte. Nota: cada horizonte predice el
mismo instante con distinto *lead time* (3, 48 y 72 h de anticipación).

In [ ]:
def mascara_mes(ts, y=2026, m=3):
    ini = np.datetime64(f"{y}-{m:02d}-01")
    fin = (np.datetime64(f"{y}-{m+1:02d}-01") if m < 12
           else np.datetime64(f"{y+1}-01-01"))
    return (ts >= ini) & (ts < fin)

if PREDS_OK:
    j = estaciones.index(ESTACION_FORECAST)
    pred_c, true_c, ts_c, _, _ = cargar_preds(RES_A3T, "a3tgcn", "corto", met_a3t)
    m = mascara_mes(ts_c)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(ts_c[m], true_c[m, j], color="black", lw=1.7, label="Observado")
    for esc in ESCENARIOS:
        pred, true, ts, L, h = cargar_preds(RES_A3T, "a3tgcn", esc, met_a3t)
        mm = mascara_mes(ts)
        ax.plot(ts[mm], pred[mm, j], lw=1.1, alpha=0.85, color=COLORS[esc],
                label=f"Pred {esc} (h={h})")
    ax.set_title(f"Forecasting temperatura — {ESTACION_FORECAST} — Marzo 2026 (A3T-GCN)")
    ax.set_ylabel("Temperatura (°C)"); ax.legend(ncol=4, loc="upper center")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "forecast_marzo2026.png"), bbox_inches="tight")
    plt.show()
else:
    print("Faltan arrays de predicción; omito el forecasting.")


### 9b. Detalle 1–7 de marzo 2026 (zoom)

In [ ]:
if PREDS_OK:
    z0, z1 = np.datetime64("2026-03-01"), np.datetime64("2026-03-08")
    j = estaciones.index(ESTACION_FORECAST)
    fig, ax = plt.subplots(figsize=(14, 5))
    mz = (ts_c >= z0) & (ts_c < z1)
    ax.plot(ts_c[mz], true_c[mz, j], color="black", lw=2.2, label="Observado")
    for esc in ESCENARIOS:
        pred, true, ts, L, h = cargar_preds(RES_A3T, "a3tgcn", esc, met_a3t)
        mm = (ts >= z0) & (ts < z1)
        ax.plot(ts[mm], pred[mm, j], lw=1.4, marker=".", ms=3,
                color=COLORS[esc], label=f"Pred {esc} (h={h})")
    ax.set_title(f"Detalle 1–7 marzo 2026 — {ESTACION_FORECAST}")
    ax.set_ylabel("Temperatura (°C)"); ax.legend(ncol=4, loc="upper center")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "forecast_marzo2026_zoom.png"), bbox_inches="tight")
    plt.show()


### 9c. Forecasting marzo 2026 — todas las estaciones (horizonte corto)

In [ ]:
if PREDS_OK:
    pred, true, ts, L, h = cargar_preds(RES_A3T, "a3tgcn", "corto", met_a3t)
    mm = mascara_mes(ts)
    fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
    for ax, est in zip(axes.ravel(), estaciones):
        jj = estaciones.index(est)
        ax.plot(ts[mm], true[mm, jj], color="black", lw=1.0, label="obs")
        ax.plot(ts[mm], pred[mm, jj], color=COLORS["corto"], lw=1.0, alpha=0.85,
                label=f"pred h={h}")
        ax.set_title(est); ax.set_ylabel("°C")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d"))
    axes.ravel()[0].legend(loc="upper right", fontsize=9)
    fig.suptitle("Observado vs predicho (h=3) — Marzo 2026 por estación", y=1.0)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "forecast_marzo2026_estaciones.png"), bbox_inches="tight")
    plt.show()


## 10. Comparación con DCRNN (opcional)
Se activa automáticamente cuando exista `resultados/dcrnn/metricas_dcrnn.json`.

In [ ]:
dcrnn_path = os.path.join(RES_DCRNN, "metricas_dcrnn.json")
if os.path.exists(dcrnn_path):
    met_d = cargar_json(dcrnn_path)
    comp = pd.DataFrame({
        "A3T-GCN MAE": {e: met_a3t[e]["global"]["MAE"] for e in ESCENARIOS},
        "DCRNN MAE":   {e: met_d[e]["global"]["MAE"]   for e in ESCENARIOS},
        "A3T-GCN R²":  {e: met_a3t[e]["global"]["R2"]  for e in ESCENARIOS},
        "DCRNN R²":    {e: met_d[e]["global"]["R2"]    for e in ESCENARIOS},
    }).round(4)
    display(comp)

    x = np.arange(len(ESCENARIOS)); w = 0.35
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    ax.bar(x - w/2, [met_a3t[e]["global"]["MAE"] for e in ESCENARIOS], w,
           label="A3T-GCN", color="#264653")
    ax.bar(x + w/2, [met_d[e]["global"]["MAE"] for e in ESCENARIOS], w,
           label="DCRNN", color="#2a9d8f")
    ax.set_xticks(x); ax.set_xticklabels(ESCENARIOS)
    ax.set_ylabel("MAE (°C)"); ax.set_title("A3T-GCN vs DCRNN — MAE por escenario")
    ax.legend()
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "comparativa_modelos.png"), bbox_inches="tight")
    plt.show()
else:
    print("DCRNN aún no disponible. Corre entrenar_dcrnn.py y re-ejecuta esta celda.")


## 11. Lectura de resultados
- El error crece con el horizonte (degradación esperada); revisa si el salto
  grande ocurre de *corto* a *medio* y luego se aplana (dominio del ciclo diurno).
- Estaciones de valle (Los Chillos, Tumbaco) suelen tener mayor MAE por su
  microclima; el R² se mantiene alto porque tienen mayor varianza diurna.
- Todas las figuras y tablas se guardan en `resultados/figuras/`.
